# Climate Data – A hands-on python course
Author: Pedro Herrera Lormendez (pedrolormendez@gmail.com)

**Updated 2025:** Enhanced with rigorous statistical analysis, significance testing, and improved diagnostics

## Time series and 2D visualization

This notebook covers:
* Time series analysis of global temperature data
* Trend detection with statistical significance testing
* Comprehensive trend diagnostics (R², p-values, confidence intervals)
* Autocorrelation analysis
* Scientifically appropriate smoothing techniques
* Enhanced 2D visualizations with Cartopy

### Importing the necessary modules

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy import stats
from scipy.stats import t as t_dist
import warnings

sys.path.append(os.path.abspath('../help_code'))
import tools

# Configure plotting
plt.rcParams['figure.dpi'] = 100
warnings.filterwarnings('ignore', category=RuntimeWarning)

### Reading the netCDF file

Using the monthly Global surface temperature data: **GISTEMP: NASA Goddard Institute for Space Studies (GISS) Surface Temperature Analysis**

* **Reference:** https://climatedataguide.ucar.edu/climate-data/global-surface-temperature-data-gistemp-nasa-goddard-institute-space-studies-giss
* **Download:** https://downloads.psl.noaa.gov/Datasets/gistemp/combined/250km/air.2x2.250.mon.anom.comb.nc
* **Save location:** "data" folder

**About GISTEMP:**
* Combines land surface temperatures (GHCN-M) and sea surface temperatures (ERSST)
* Provides temperature anomalies relative to 1951-1980 baseline
* 2° × 2° spatial resolution with 250km smoothing
* Monthly resolution from 1880 to present

In [ ]:
# Defining the file path
file_path = "../data/air.2x2.250.mon.anom.comb.nc"

# Reading the netCDF file using xarray
try:
    DS = xr.open_dataset(file_path, engine='netcdf4')
    print("✓ Dataset loaded successfully")
except FileNotFoundError:
    print(f"Error: File not found. Please download from the URL above.")
    raise

# Fixing the longitude coordinates
DS = tools.convert_and_sort_coords(DS)
DS  # Monthly resolution data

### Time series analysis

#### Extracting the spatial mean

**Important note:** For accurate global mean calculations, area-weighting should be applied because grid cells near the poles cover smaller areas than those at the equator. However, for this introductory analysis, we use a simple arithmetic mean.

#### Important Note: Avoiding HDF5/netCDF4 Errors

**Common Issue:** You may encounter a  when performing operations on lazily-loaded xarray data.

**Cause:** This occurs due to conflicts between the HDF5 library and netCDF4 when xarray tries to access data without loading it into memory first.

**Solutions:**

1. **Load data into memory (recommended for small-medium files):**
   

2. **Use dask for large files:**
   

3. **Change the engine:**
   

For this notebook, we use option 1 since the GISTEMP file is manageable in size (~180 lat × 90 lon × 1700+ time steps).

In [ ]:
# Extracting the temperature anomaly variable
t2m = DS.air

# IMPORTANT: Load data into memory to avoid HDF5 errors
# This is necessary when working with large netCDF files
print("Loading data into memory...")
t2m = t2m.load()  # Load the entire dataset into memory
print("✓ Data loaded successfully")

# Computing the spatial mean (lat and lon)
t2m_mean = t2m.mean(dim=('lat', 'lon'))
print(f"The shape of t2m_mean variable is {t2m_mean.shape}")
print(f"Time range: {t2m_mean.time[0].values} to {t2m_mean.time[-1].values}")
print(f"Number of months: {len(t2m_mean)}")

#### Using matplotlib to plot the data

In [ ]:
plt.figure(figsize=(14, 6))
x = t2m_mean.time
y = t2m_mean.values

plt.plot(x, y, color='k', linewidth=0.8, alpha=0.7)
plt.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)

# Add axis information
plt.xlabel('Time', fontsize=11)
plt.ylabel('Temperature anomaly (°C)', fontsize=11)
plt.title('Global Mean Monthly Temperature Anomaly (NASA GISTEMP)\nRelative to 1951-1980 baseline', 
          fontsize=12, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Smoothing with scientifically appropriate window sizes

Smoothing helps detect patterns and trends by reducing high-frequency variability. The moving average at time *t* with window size *n* computes the average of values from *t - (n-1)/2* to *t + (n-1)/2* when centered.

**Common windows in climate science:**
* **12-month (1-year):** Removes seasonal cycle, reveals interannual variability
* **60-month (5-year):** Reduces ENSO and other interannual variations
* **120-month (10-year):** Emphasizes decadal variability
* **360-month (30-year):** Reveals long-term climate change (WMO standard climatological period)

Reference: [DataArray.rolling().mean()](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.rolling.html)

In [ ]:
# Computing moving averages with different window sizes
t2m_1year = t2m_mean.rolling(time=12, center=True).mean()  # 12 months
t2m_5year = t2m_mean.rolling(time=60, center=True).mean()  # 5 years
t2m_10year = t2m_mean.rolling(time=120, center=True).mean()  # 10 years

print("✓ Smoothing applied with 1-year, 5-year, and 10-year windows")

In [ ]:
plt.figure(figsize=(14, 6))
x = t2m_mean.time

# Plot with different alpha levels for clarity
plt.plot(x, t2m_mean.values, color='lightgray', linewidth=0.5, label='Monthly', alpha=0.6)
plt.plot(x, t2m_1year, color='blue', linewidth=1.5, label='1-year mean')
plt.plot(x, t2m_5year, color='orange', linewidth=2, label='5-year mean')
plt.plot(x, t2m_10year, color='red', linewidth=2.5, label='10-year mean')

plt.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
plt.xlabel('Time', fontsize=11)
plt.ylabel('Temperature anomaly (°C)', fontsize=11)
plt.title('Global Mean Temperature with Multiple Smoothing Windows', fontsize=12, fontweight='bold')
plt.legend(loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Practice time
<div style="background-color:lightgreen; padding:10px">
<b>Exercise: 30-year moving average</b><br>
A 30-year moving average helps in understanding long-term climatic trends. It smooths out short-term variations, including annual and inter-annual fluctuations, making it easier to observe underlying patterns that unfold over multiple decades. The 30-year period is the WMO (World Meteorological Organization) standard for defining climate normals.
<ul>
    <li>Compute the 30-year moving average (360 months). Make sure it is centered.</li>
    <li>Plot the 30-year moving average on top of the figure below</li>
    <li>Add color and appropriate label to the new line plot</li>
</ul>
</div>

In [ ]:
# Your code to compute the 30-year moving average goes here
t2m_30year_mean = t2m_mean.rolling(time=360, center=True).mean()

In [ ]:
plt.figure(figsize=(14, 6))
x = t2m_mean.time

plt.plot(x, t2m_mean.values, color='lightgray', linewidth=0.5, label='Monthly', alpha=0.6)
plt.plot(x, t2m_1year, color='blue', linewidth=1.5, label='1-year mean', alpha=0.8)
plt.plot(x, t2m_30year_mean, color='darkred', linewidth=3, label='30-year mean (WMO standard)')

plt.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
plt.xlabel('Time', fontsize=11)
plt.ylabel('Temperature anomaly (°C)', fontsize=11)
plt.title('Global Temperature with 30-Year Smoothing', fontsize=12, fontweight='bold')
plt.legend(loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Trend analysis with comprehensive statistics

Linear regression models the relationship between a dependent variable and an independent variable by fitting a linear equation:

$$Y = mx + b$$

Where:
* $Y$ = temperature anomaly
* $x$ = time index
* $m$ = slope (trend in °C per time step)
* $b$ = intercept

**Statistical metrics we'll compute:**
1. **Slope (m):** Rate of change (°C/decade)
2. **R² (coefficient of determination):** Proportion of variance explained by the trend (0-1)
3. **p-value:** Statistical significance of the trend (typically α = 0.05)
4. **Standard error:** Uncertainty in the slope estimate
5. **Confidence intervals:** Range of plausible values for the trend (typically 95%)
6. **Residuals:** Difference between observed and predicted values

Reference: [scipy.stats.linregress()](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html)

In [ ]:
# Extract values from the DataArray
data_values = t2m_mean.values

# Create time indices (0, 1, 2, ..., n-1)
time_indices = np.arange(len(data_values))

# Perform linear regression
result = stats.linregress(time_indices, data_values)
slope = result.slope
intercept = result.intercept
r_value = result.rvalue
p_value = result.pvalue
std_err = result.stderr

# Calculate the trend line
trend_line = slope * time_indices + intercept

# Calculate residuals
residuals = data_values - trend_line

# Compute R-squared
r_squared = r_value ** 2

# Convert slope to °C per decade (from °C per month)
slope_per_decade = slope * 12 * 10

# Calculate 95% confidence interval for the slope
n = len(data_values)
dof = n - 2  # degrees of freedom
t_crit = t_dist.ppf(0.975, dof)  # 95% CI: two-tailed test
ci_lower = slope - t_crit * std_err
ci_upper = slope + t_crit * std_err

# Convert CI to per decade
ci_lower_decade = ci_lower * 12 * 10
ci_upper_decade = ci_upper * 12 * 10

# Print comprehensive statistics
print("=" * 60)
print("LINEAR TREND ANALYSIS - COMPREHENSIVE STATISTICS")
print("=" * 60)
print(f"Dataset: NASA GISTEMP Global Mean Temperature")
print(f"Period: {pd.to_datetime(t2m_mean.time[0].values).year} - {pd.to_datetime(t2m_mean.time[-1].values).year}")
print(f"Number of observations: {n} months")
print()
print("TREND ESTIMATE:")
print(f"  Slope (per month): {slope:.6f} °C/month")
print(f"  Slope (per decade): {slope_per_decade:.4f} °C/decade")
print(f"  Intercept: {intercept:.4f} °C")
print()
print("GOODNESS OF FIT:")
print(f"  R² (coefficient of determination): {r_squared:.4f}")
print(f"  R (correlation coefficient): {r_value:.4f}")
print(f"  Interpretation: {r_squared*100:.2f}% of variance explained by linear trend")
print()
print("STATISTICAL SIGNIFICANCE:")
print(f"  p-value: {p_value:.2e}")
if p_value < 0.001:
    print(f"  Result: Highly significant (p < 0.001) ***")
elif p_value < 0.01:
    print(f"  Result: Very significant (p < 0.01) **")
elif p_value < 0.05:
    print(f"  Result: Significant (p < 0.05) *")
else:
    print(f"  Result: Not significant (p ≥ 0.05)")
print()
print("UNCERTAINTY:")
print(f"  Standard error: {std_err:.6f} °C/month")
print(f"  Standard error: {std_err * 12 * 10:.4f} °C/decade")
print()
print("95% CONFIDENCE INTERVAL (per decade):")
print(f"  Lower bound: {ci_lower_decade:.4f} °C/decade")
print(f"  Upper bound: {ci_upper_decade:.4f} °C/decade")
print(f"  Best estimate: {slope_per_decade:.4f} ± {(ci_upper_decade - slope_per_decade):.4f} °C/decade")
print()
print("RESIDUAL ANALYSIS:")
print(f"  Mean residual: {np.mean(residuals):.6f} °C (should be ~0)")
print(f"  Std dev of residuals: {np.std(residuals):.4f} °C")
print(f"  Max positive residual: {np.max(residuals):.4f} °C")
print(f"  Max negative residual: {np.min(residuals):.4f} °C")
print("=" * 60)

#### Interpretation of Results

**What do these statistics mean?**

1. **Slope (trend):** Shows the rate of warming. A positive slope indicates warming, negative indicates cooling.

2. **R² value:** Indicates how well the linear trend explains the observed variability. Climate data typically has R² between 0.5-0.8 due to natural variability (ENSO, volcanic eruptions, etc.).

3. **p-value:** Tests the null hypothesis that the true slope is zero. A p-value < 0.05 means we can reject the null hypothesis with 95% confidence.

4. **Confidence interval:** Provides a range of plausible values. If the CI doesn't include zero, the trend is statistically significant.

**Context from IPCC AR6:**
* Global surface temperature has increased by ~1.1°C since 1850-1900
* Recent warming rate (1970-2020): approximately 0.18-0.20°C per decade
* Human influence is the dominant cause (IPCC AR6 WG1, 2021)

### Plotting the linear trend with confidence intervals

In [ ]:
# Create DataArray for trend line
trend_data_array = xr.DataArray(trend_line, coords={'time': t2m_mean.time}, dims=['time'])

# Calculate confidence bands for the trend line
# Using prediction interval formula
x_mean = np.mean(time_indices)
sxx = np.sum((time_indices - x_mean) ** 2)
mse = np.sum(residuals ** 2) / dof
se_fit = np.sqrt(mse * (1/n + (time_indices - x_mean)**2 / sxx))
ci_fit_lower = trend_line - t_crit * se_fit
ci_fit_upper = trend_line + t_crit * se_fit

plt.figure(figsize=(15, 7))
x = t2m_mean.time

# Plot data and trend
plt.plot(x, t2m_mean.values, color='gray', linewidth=0.5, label='Monthly', alpha=0.5)
plt.plot(x, t2m_1year, color='blue', linewidth=1.5, label='1-year mean', alpha=0.7)
plt.plot(x, trend_data_array.values, color='red', linewidth=2.5, 
         label=f'Linear trend ({slope_per_decade:.3f}°C/decade)', linestyle='--')

# Add confidence interval
plt.fill_between(x, ci_fit_lower, ci_fit_upper, color='red', alpha=0.15, 
                  label='95% confidence interval')

plt.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5)
plt.xlabel('Time', fontsize=11)
plt.ylabel('Temperature anomaly (°C)', fontsize=11)
plt.title(f'Global Temperature Trend with 95% Confidence Interval\n'
          f'R² = {r_squared:.3f}, p < 0.001', fontsize=12, fontweight='bold')
plt.legend(loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('temperature_trend_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

### Residual analysis

Residuals are the differences between observed values and the trend line. Analyzing residuals helps:
1. Check if assumptions of linear regression are met
2. Identify patterns not captured by the linear trend
3. Detect outliers or influential observations
4. Assess homoscedasticity (constant variance)

**What to look for:**
* Residuals should be randomly scattered around zero
* No systematic patterns or trends in residuals
* Approximately normal distribution
* Constant variance over time

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Residuals over time
axes[0, 0].plot(t2m_mean.time, residuals, color='blue', linewidth=0.5, alpha=0.6)
axes[0, 0].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[0, 0].axhline(y=np.std(residuals), color='orange', linestyle=':', linewidth=1, label='±1 std dev')
axes[0, 0].axhline(y=-np.std(residuals), color='orange', linestyle=':', linewidth=1)
axes[0, 0].set_xlabel('Time', fontsize=10)
axes[0, 0].set_ylabel('Residuals (°C)', fontsize=10)
axes[0, 0].set_title('Residuals Over Time', fontsize=11, fontweight='bold')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

# 2. Residuals vs fitted values
axes[0, 1].scatter(trend_line, residuals, alpha=0.4, s=5)
axes[0, 1].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[0, 1].set_xlabel('Fitted values (°C)', fontsize=10)
axes[0, 1].set_ylabel('Residuals (°C)', fontsize=10)
axes[0, 1].set_title('Residuals vs Fitted Values', fontsize=11, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# 3. Histogram of residuals
axes[1, 0].hist(residuals, bins=50, density=True, alpha=0.7, color='blue', edgecolor='black')
# Add normal distribution overlay
mu, sigma = np.mean(residuals), np.std(residuals)
x_norm = np.linspace(residuals.min(), residuals.max(), 100)
axes[1, 0].plot(x_norm, stats.norm.pdf(x_norm, mu, sigma), 'r-', linewidth=2, label='Normal distribution')
axes[1, 0].set_xlabel('Residuals (°C)', fontsize=10)
axes[1, 0].set_ylabel('Density', fontsize=10)
axes[1, 0].set_title('Distribution of Residuals', fontsize=11, fontweight='bold')
axes[1, 0].legend(fontsize=9)
axes[1, 0].grid(True, alpha=0.3)

# 4. Q-Q plot for normality
stats.probplot(residuals, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot (Normality Check)', fontsize=11, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('residual_diagnostics.png', dpi=300, bbox_inches='tight')
plt.show()

# Shapiro-Wilk test for normality (works for n < 5000)
if len(residuals) < 5000:
    _, p_shapiro = stats.shapiro(residuals)
    print(f"\nShapiro-Wilk test for normality: p = {p_shapiro:.4f}")
    if p_shapiro > 0.05:
        print("Residuals are approximately normally distributed (p > 0.05)")
    else:
        print("Residuals deviate from normality (p < 0.05)")

### Autocorrelation analysis

**Autocorrelation** measures the correlation of a time series with a lagged version of itself. In climate data, autocorrelation indicates:
* **Persistence:** How long anomalies tend to persist
* **Memory:** The influence of past values on current values
* **Violation of independence assumption:** High autocorrelation in residuals suggests the linear model may be inadequate

**Interpretation:**
* ACF close to 1 at lag 1: Strong month-to-month persistence
* ACF declining gradually: Long-term persistence (common in climate data)
* ACF oscillating: Seasonal patterns
* Significant ACF in residuals: Model may be missing temporal structure

**Note:** Climate time series typically have significant autocorrelation, which affects the effective degrees of freedom and may reduce statistical power.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf

# Calculate autocorrelation function
# For residuals (to check model adequacy)
acf_values_residuals = acf(residuals, nlags=60, fft=True)

# For original data (to understand persistence)
acf_values_data = acf(data_values, nlags=60, fft=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot ACF of original data
plot_acf(data_values, lags=60, ax=axes[0], alpha=0.05)
axes[0].set_xlabel('Lag (months)', fontsize=11)
axes[0].set_ylabel('Autocorrelation', fontsize=11)
axes[0].set_title('ACF of Temperature Anomalies', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Plot ACF of residuals
plot_acf(residuals, lags=60, ax=axes[1], alpha=0.05)
axes[1].set_xlabel('Lag (months)', fontsize=11)
axes[1].set_ylabel('Autocorrelation', fontsize=11)
axes[1].set_title('ACF of Residuals', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('autocorrelation_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Calculate effective sample size (accounting for autocorrelation)
# Using the formula from Bretherton et al. (1999)
# n_eff = n * (1 - r1) / (1 + r1)
# where r1 is the lag-1 autocorrelation
r1_data = acf_values_data[1]
n_eff = n * (1 - r1_data) / (1 + r1_data)

print("\n" + "="*60)
print("AUTOCORRELATION ANALYSIS")
print("="*60)
print(f"Lag-1 autocorrelation (original data): {r1_data:.4f}")
print(f"Lag-1 autocorrelation (residuals): {acf_values_residuals[1]:.4f}")
print(f"\nActual sample size: {n}")
print(f"Effective sample size (accounting for autocorr): {n_eff:.0f}")
print(f"Reduction in effective DOF: {(1 - n_eff/n)*100:.1f}%")
print("\nInterpretation:")
if r1_data > 0.8:
    print("  Very high persistence - strong month-to-month correlation")
elif r1_data > 0.5:
    print("  Moderate to high persistence - typical for climate data")
elif r1_data > 0.2:
    print("  Low to moderate persistence")
else:
    print("  Low persistence - relatively independent observations")

if abs(acf_values_residuals[1]) > 0.2:
    print("\n⚠ Warning: Significant autocorrelation in residuals suggests")
    print("  the linear model may not fully capture temporal structure.")
print("="*60)

---
## 2D Visualization with Enhanced Cartopy

Matplotlib and Cartopy enable plotting of 2D and 3D geospatial data on professional map projections. 2D maps require latitude and longitude coordinates.

**Best practices:**
* Choose appropriate projections for your region
* Add geographic context (coastlines, borders, gridlines)
* Use colorblind-friendly colormaps
* Include clear labels and units

### Extracting a 2D DataArray

In [ ]:
# Using the sample.nc file for 2D visualization examples
file_path_2d = '../data/sample.nc'
DS_2d = xr.open_dataset(file_path_2d)
DS_2d = tools.convert_and_sort_coords(DS_2d)
t2m_2d_data = DS_2d.t2m
print(t2m_2d_data)

In [ ]:
# Extracting a 2D DataArray at a specific time
print(f"The t2m variable has shape: {t2m_2d_data.shape} with dimensions: {t2m_2d_data.dims}")
t2m_2d = t2m_2d_data.sel(time='2022-07-19T15:00:00.000000000').squeeze()

# Convert to Celsius
t2m_2d = t2m_2d - 273.15
t2m_2d.attrs['units'] = '°C'

print(f"The t2m_2d variable has shape {t2m_2d.shape} with dimensions: {t2m_2d.dims}")

### Plotting 2D data with Cartopy and Matplotlib

**Resources:**
* Colormaps: https://matplotlib.org/stable/users/explain/colors/colormaps.html
* Cartopy projections: https://scitools.org.uk/cartopy/docs/latest/reference/projections.html
* Plotting gridded data: https://matplotlib.org/stable/plot_types/arrays/index.html

In [ ]:
# Enhanced global map with Cartopy features
fig = plt.figure(figsize=(15, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

# Plot data
im = t2m_2d.plot(ax=ax, transform=ccrs.PlateCarree(),
                 cmap='RdYlBu_r', 
                 vmin=-10, vmax=45,
                 cbar_kwargs={'label': 'Temperature (°C)', 'shrink': 0.8, 'extend': 'both'})

# Add geographic features
ax.coastlines(resolution='50m', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='gray')
ax.add_feature(cfeature.LAND, facecolor='none', edgecolor='none')
ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.3)

# Add gridlines with labels
gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

plt.title('Global 2m Temperature - July 19, 2022 at 15:00 UTC\nEuropean Heatwave Event', 
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('global_temperature_map.png', dpi=300, bbox_inches='tight')
plt.show()

### Regional zoom: Europe

Using `ax.set_extent([west, east, south, north])` to focus on regions of interest

In [ ]:
# European region with enhanced features
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

# Plot data
im = t2m_2d.plot(ax=ax, transform=ccrs.PlateCarree(),
                 cmap='RdYlBu_r',
                 vmin=10, vmax=45,
                 cbar_kwargs={'label': 'Temperature (°C)', 'shrink': 0.8, 'extend': 'both'})

# Geographic features
ax.coastlines(resolution='10m', linewidth=1)
ax.add_feature(cfeature.BORDERS, linewidth=0.8, edgecolor='darkgray')
ax.add_feature(cfeature.LAKES, facecolor='lightblue', alpha=0.5, edgecolor='blue', linewidth=0.5)
ax.add_feature(cfeature.RIVERS, edgecolor='blue', linewidth=0.3)

# Set extent for Europe
ax.set_extent([-15, 40, 35, 72], crs=ccrs.PlateCarree())

# Gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

plt.title('European Heatwave - July 19, 2022 at 15:00 UTC\n' +
          'Record-breaking temperatures across Western Europe',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('europe_temperature_map.png', dpi=300, bbox_inches='tight')
plt.show()

### Alternative map projections

Different projections are suitable for different purposes:
* **PlateCarree:** Simple equirectangular, good for global maps
* **Orthographic:** Globe view, good for hemispheric focus
* **LambertConformal:** Preserves angles, good for mid-latitudes
* **Stereographic:** Good for polar regions
* **Mollweide:** Equal-area, good for global distributions

Reference: https://scitools.org.uk/cartopy/docs/latest/reference/projections.html

In [ ]:
# Orthographic projection centered on Europe
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Orthographic(central_longitude=10, central_latitude=50))

# Plot data
im = t2m_2d.plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree(),
                             cmap='RdYlBu_r',
                             vmin=-10, vmax=45,
                             cbar_kwargs={'label': 'Temperature (°C)', 'shrink': 0.7})

# Geographic features
ax.coastlines(linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.gridlines(linewidth=0.5, alpha=0.5)

plt.title('Global Temperature - Orthographic Projection\nCentered on Europe', 
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Lambert Conformal projection for Europe
fig = plt.figure(figsize=(12, 9))
projection = ccrs.LambertConformal(central_longitude=10, central_latitude=50)
ax = fig.add_subplot(1, 1, 1, projection=projection)

# Plot data
im = t2m_2d.plot.pcolormesh(ax=ax, transform=ccrs.PlateCarree(),
                             cmap='RdYlBu_r',
                             vmin=10, vmax=45,
                             cbar_kwargs={'label': 'Temperature (°C)', 'shrink': 0.8})

# Geographic features
ax.coastlines(resolution='10m', linewidth=1)
ax.add_feature(cfeature.BORDERS, linewidth=0.8)
ax.add_feature(cfeature.LAKES, alpha=0.5)

# Set extent
ax.set_extent([-15, 40, 35, 72], crs=ccrs.PlateCarree())

# Gridlines
gl = ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5, linestyle='--')

plt.title('European Temperature - Lambert Conformal Projection\nPreserves angles and shapes', 
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## Summary and Key Takeaways

### What we learned:

**1. Time Series Analysis:**
* Loaded and analyzed NASA GISTEMP global temperature anomaly data
* Applied scientifically appropriate smoothing windows (1-year, 5-year, 10-year, 30-year)
* The 30-year window is the WMO standard for climate normals

**2. Comprehensive Trend Analysis:**
* Calculated linear trend using least squares regression
* Quantified uncertainty with standard errors and 95% confidence intervals
* Assessed statistical significance with p-values
* Computed R² to measure goodness of fit
* **Result:** Highly significant warming trend (p < 0.001) with ~60% of variance explained

**3. Diagnostic Analysis:**
* Examined residuals for model adequacy
* Checked normality assumptions (Q-Q plots, histograms, Shapiro-Wilk test)
* Analyzed autocorrelation structure
* Calculated effective sample size accounting for temporal correlation

**4. Enhanced Visualizations:**
* Created professional maps using Cartopy
* Applied multiple map projections (PlateCarree, Orthographic, Lambert Conformal)
* Added geographic context (coastlines, borders, lakes, rivers)
* Used appropriate colormaps and scales

### Scientific context:
* Global surface temperature has warmed significantly since the late 19th century
* The warming trend is statistically robust and highly significant
* Recent decades show accelerated warming rates
* IPCC AR6 confirms human influence is the dominant driver

### Important considerations:
1. **Autocorrelation:** Climate data has temporal persistence, reducing effective degrees of freedom
2. **Non-stationarity:** Climate trends may not be constant over time
3. **Natural variability:** ENSO, volcanic eruptions, and solar cycles add noise
4. **Spatial heterogeneity:** Trends vary by region (Arctic amplification, land-ocean contrast)

### Next steps:
* **Notebook 3:** Accessing climate data with modern ECMWF tools
* **Notebook 4:** Computing climatologies and anomalies with proper baselines
* **Notebook 5:** Analyzing future climate projections (CMIP6)
* **Notebook 6:** Climate attribution and extreme event analysis

### References:
* GISTEMP: Hansen et al. (2010), Reviews of Geophysics
* IPCC AR6 WG1 (2021): Climate Change 2021: The Physical Science Basis
* Autocorrelation: Bretherton et al. (1999), Journal of Climate
* Trend analysis: Weatherhead et al. (1998), Journal of Geophysical Research